# Olist Marketplace Intelligence
A reproducible SQL + Python investigation of growth, fulfillment and customer experience.

Question: where should a marketplace operations team intervene first? Data is historical (2016–2018), not current company performance. Original data and license: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [1]:
from pathlib import Path
import sqlite3, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data/marketplace.sqlite').exists():
    ROOT = ROOT.parent
db = sqlite3.connect(ROOT / 'data/marketplace.sqlite')
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", db).to_string(index=False))


                name
category_translation
           customers
           fact_item
          fact_order
         geolocation
         order_items
              orders
            payments
            products
             reviews
             sellers


## 1. Validate the grain before calculating KPIs
Every source order must appear exactly once. Child-table aggregation is implemented in sql/01_order_mart.sql. Full missingness and integrity checks are in docs/DATA_QUALITY.html.

In [2]:
print(pd.read_sql_query("""SELECT COUNT(*) rows,COUNT(DISTINCT order_id) unique_orders,
 SUM(gmv) item_value,SUM(freight) freight,SUM(payment_value) payment_value FROM fact_order""",db).to_string(index=False))
audit=json.loads((ROOT/'data/processed/quality_report.json').read_text())
print(pd.DataFrame(audit['checks'])[['check','exceptions','severity']].to_string(index=False))


 rows  unique_orders  item_value    freight  payment_value
99441          99441  13591643.7 2251909.54    16008872.12
                                    check  exceptions severity
             orders duplicate primary key           0     HIGH
          customers duplicate primary key           0     HIGH
           products duplicate primary key           0     HIGH
            sellers duplicate primary key           0     HIGH
        order_items duplicate primary key           0     HIGH
   orders.customer_id orphan to customers           0     HIGH
    order_items.order_id orphan to orders           0     HIGH
       payments.order_id orphan to orders           0     HIGH
        reviews.order_id orphan to orders           0     HIGH
order_items.product_id orphan to products           0     HIGH
  order_items.seller_id orphan to sellers           0     HIGH
          reviews multiple rows per order         551   MEDIUM
                  nonpositive item prices           0     HIGH


## 2. Define the executive scorecard
Delivered GMV is item merchandise value, not company revenue. Late rate compares calendar dates among delivered orders with both dates recorded.

In [3]:
f=pd.read_sql_query("SELECT * FROM fact_order WHERE purchase_month BETWEEN '2017-01' AND '2018-08'",db)
print(json.dumps(json.loads((ROOT/'data/processed/headline_metrics.json').read_text()),indent=2))


{
  "orders": 99092,
  "delivered_orders": 96211,
  "delivered_gmv": 13181027.13,
  "aov": 137.0,
  "eligible": 96203,
  "late_orders": 6531,
  "late_rate": 0.06788769580990198,
  "review_coverage": 0.9933162167499974,
  "low_rating_late": 0.6241768579492004,
  "low_rating_ontime": 0.0924738175865085,
  "median_days": 10.209988425951451,
  "single_seller_coverage": 0.9867779591073044,
  "comparable_ytd_growth": 1.4113014544161704,
  "repeat90": 0.02008245072177521,
  "repeat90_eligible": 75439
}


## 3. Compare like-for-like growth
Both years use Jan–Aug delivered purchases. Comparing all of 2017 with only part of 2018 would mislead.

In [4]:
print(pd.read_sql_query("""SELECT substr(purchase_month,1,4) year,COUNT(*) delivered_orders,SUM(gmv) gmv
 FROM fact_order WHERE order_status='delivered' AND purchase_month BETWEEN '2017-01' AND '2018-08'
 AND substr(purchase_month,6,2)<='08' GROUP BY 1""",db).to_string(index=False))


year  delivered_orders        gmv
2017             21998 2993456.13
2018             52783 7218125.12


## 4. Diagnose route concentration
Rank by late count as well as rate; tiny high-rate segments should not dominate the improvement queue.

In [5]:
e=f[f.delivery_eligible.eq(1)]
state=e.groupby('customer_state').agg(eligible=('order_id','size'),late=('is_late','sum'),late_rate=('is_late','mean'))
state['late_share']=state.late/state.late.sum()
print(state.sort_values('late',ascending=False).head(10).to_string())


                eligible    late  late_rate  late_share
customer_state                                         
SP                 40399  1817.0   0.044976    0.278212
RJ                 12310  1495.0   0.121446    0.228908
MG                 11319   519.0   0.045852    0.079467
BA                  3253   396.0   0.121734    0.060634
RS                  5327   325.0   0.061010    0.049763
SC                  3537   291.0   0.082273    0.044557
ES                  1992   214.0   0.107430    0.032767
PR                  4903   199.0   0.040587    0.030470
CE                  1273   176.0   0.138256    0.026948
PE                  1587   153.0   0.096408    0.023427


## 5. Quantify customer experience association
Latest review per order; low rating is 1–2 stars. This is descriptive and potentially confounded. It does not measure the impact of a communication intervention.

In [6]:
rated=e[e.review_score.notna()].copy()
rated['low']=rated.review_score.le(2)
print(rated.groupby('is_late').agg(rated_orders=('order_id','size'),low_ratings=('low','sum'),low_rating_rate=('low','mean'),avg_score=('review_score','mean')).to_string())


         rated_orders  low_ratings  low_rating_rate  avg_score
is_late                                                       
0.0             89182         8247         0.092474   4.291157
1.0              6378         3981         0.624177   2.270618


## 6. Avoid incomplete retention windows
Identity uses customer_unique_id. Only first-observed delivered purchasers with 90 days of follow-up opportunity are included. The source does not reveal purchases outside Olist marketplaces.

In [7]:
print(pd.read_csv(ROOT/'data/processed/cohort_90_day.csv').to_string(index=False))


 cohort  customers  repeat90
2017-01        717  0.030683
2017-02       1628  0.015971
2017-03       2503  0.022373
2017-04       2256  0.018617
2017-05       3451  0.024920
2017-06       3037  0.025354
2017-07       3752  0.021855
2017-08       4057  0.023416
2017-09       4004  0.024975
2017-10       4328  0.019871
2017-11       7060  0.020255
2017-12       5338  0.015174
2018-01       6842  0.019000
2018-02       6288  0.025922
2018-03       6774  0.016977
2018-04       6582  0.015953
2018-05       6506  0.015524
2018-06        316  0.015823


## 7. Reject a weak advance risk flag
State × category late history, smoothed with 50 pseudo-orders. Training labels must already be observed before July 2018. Holdout is July–August purchases. Ranking below the baseline is a failed test, not a successful model. Outcome-maturation selection and time drift need further work.

In [8]:
print(json.dumps(json.loads((ROOT/'data/processed/risk_validation.json').read_text()),indent=2))
db.close()


{
  "training_orders": 81991,
  "test_orders": 12507,
  "test_base_late_rate": 0.04805309026944911,
  "flagged_orders": 2530,
  "flagged_late_rate": 0.04308300395256917,
  "late_capture": 0.18136439267886856,
  "threshold": 0.09140494078618384,
  "method": "State \u00d7 primary-category late rate; additive smoothing equivalent to 50 orders at training base rate. Train observed outcomes through June 30 2018; test July-August purchases. Top quintile threshold uses test feature scores only; ties may exceed 20%. Not a production calibrated probability model."
}


## 8. Recommendation and measurement
Audit RJ lanes; investigate high-volume seller exceptions; run a randomized proactive-communication pilot. Track late rate, low ratings, response coverage and contact cost. Use the dashboard scenario only as a planning assumption. No realized savings or uplift is claimed.

See docs/FINDINGS.md for the full decision brief and docs/METRIC_CONTRACT.md for definitions.